In [ ]:
import pandas as pd
import json
import sqlite3

**BAGIAN A - MEMBACA DATA**

In [ ]:
#Membaca file CSV menjadi DataFrame bernama df_populasi
df_populasi = pd.read_csv("populasi_asean.csv")

#Menampilkan 5 baris pertama untuk mengecek apakah data berhasil terbaca
df_populasi.head()

,kode_negara,tahun,populasi
0,BRN,2015,422212
1,BRN,2016,427564
2,BRN,2017,432772
3,BRN,2018,437810
4,BRN,2019,442680


In [ ]:
#Membaca indikator_asean.json menjadi DataFrame df_indikator
df_indikator = pd.read_json("indikator_asean.json")

#Menampilkan 5 baris pertama untuk memastikan datanya terbaca
df_indikator.head()

,iso3,tahun,pdb_per_kapita_usd,internet_persen
0,BRN,2015,30625.13,71.20
1,BRN,2016,26663.30,90.00
2,BRN,2017,28024.38,94.87
3,BRN,2018,30988.12,95.00
4,BRN,2019,30426.57,95.00


In [ ]:
#Membaca file Excel profil_negara_asean.xlsx
df_profil = pd.read_excel(
    "profil_negara_asean.xlsx",
    sheet_name="profil" #Mengambil data dari sheet yang namanya profil
)

df_profil.head() #Menyimpan data ke variabel df_profil

,iso3,nama_negara,ibu_kota,tgl_bergabung_asean,luas_km2,mata_uang
0,BRN,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND
1,IDN,Indonesia,Jakarta,1967-08-08,1877519,IDR
2,KHM,Kamboja,Phnom Penh,1999-04-30,176520,KHR
3,LAO,Laos,Vientiane,1997-07-23,230800,LAK
4,MMR,Myanmar,Naypyidaw,1997-07-23,652670,MMK


In [ ]:
# 2. Menampilkan ukuran (shape), kolom, dan tipe data
print("--- DF POPULASI (CSV) ---")
print("Shape:", df_populasi.shape)
print(df_populasi.dtypes)

print("\n--- DF INDIKATOR (JSON) ---")
print("Shape:", df_indikator.shape)
print(df_indikator.dtypes)

print("\n--- DF PROFIL (Excel) ---")
print("Shape:", df_profil.shape)
print(df_profil.dtypes)

--- DF POPULASI (CSV) ---
Shape: (110, 3)
kode_negara    object
tahun           int64
populasi        int64
dtype: object

--- DF INDIKATOR (JSON) ---
Shape: (110, 4)
iso3                   object
tahun                   int64
pdb_per_kapita_usd    float64
internet_persen       float64
dtype: object

--- DF PROFIL (Excel) ---
Shape: (11, 6)
iso3                           object
nama_negara                    object
ibu_kota                       object
tgl_bergabung_asean    datetime64[ns]
luas_km2                        int64
mata_uang                      object
dtype: object


Penjelasan Kunci di Setiap File:

* populasi_asean.csv: Kolom kunci gabungan adalah kode_negara dan tahun.

* indikator_asean.json: Kolom kunci gabungan adalah iso3 dan tahun.

* profil_negara_asean.xlsx: Kolom kunci adalah iso3

**BAGIAN B - INTEGRASI**

1. Menyeragamkan kolom kunci

In [ ]:
# Mengubah nama kolom `kode_negara` menjadi `iso3` agar memiliki nama kunci yang sama dengan dataset indikator.
df_populasi = df_populasi.rename(
    columns={"kode_negara": "iso3"}
)

2. Menggabungkan Data Populasi dan Indikator

In [ ]:
#Menggabungkan data populasi dan indikator berdasarkan kode negara (`iso3`) dan tahun.
df_integrasi = pd.merge(
    df_populasi,
    df_indikator,
    on=["iso3", "tahun"],
    how="outer",
    validate="one_to_one",
    indicator=True
)



In [ ]:
#Menghitung jumlah data berdasarkan sumbernya untuk mengetahui data yang terdapat di kedua dataset atau hanya salah satunya.
df_integrasi["_merge"].value_counts()


,count
_merge,
both,110
left_only,0
right_only,0


In [ ]:
# Laporan hasil kecocokan baris
match_report = df_integrasi["_merge"].value_counts()
print("Hasil Validasi Merge 1:")
print(f"Cocok keduanya (both) : {match_report.get('both', 0)} baris")
print(f"Hanya di kiri (left_only) : {match_report.get('left_only', 0)} baris")
print(f"Hanya di kanan (right_only): {match_report.get('right_only', 0)} baris")

Hasil Validasi Merge 1:
Cocok keduanya (both) : 110 baris
Hanya di kiri (left_only) : 0 baris
Hanya di kanan (right_only): 0 baris


In [ ]:
df_integrasi = df_integrasi.drop(columns="_merge")

3. Menggabungkan dengan profil negara

In [ ]:
#Menggabungkan data hasil integrasi dengan profil negara berdasarkan kode `iso3`.
df_terintegrasi = pd.merge(
    df_integrasi,
    df_profil,
    on="iso3",
    how="left",
    validate="many_to_one"
)

In [ ]:
# Verifikasi total baris
print("Ukuran Data Terintegrasi:", df_terintegrasi.shape)

Ukuran Data Terintegrasi: (110, 10)


Hasil integrasi diperiksa untuk memastikan terdapat 110 baris data, yaitu 11 negara ASEAN selama 10 tahun. Duplikasi berdasarkan kombinasi negara dan tahun juga diperiksa.

4. Menambahkan tiga kolom turunan

In [ ]:
#Menghitung PDB total setiap negara dalam satuan miliar USD dari populasi dan PDB per kapita.
df_terintegrasi["PDB_total_miliar_usd"] = (
    df_terintegrasi["populasi"] *
    df_terintegrasi["pdb_per_kapita_usd"]
    / 10**9
)
#Menghitung kepadatan penduduk berdasarkan jumlah populasi dan luas wilayah.
df_terintegrasi["kepadatan_per_km2"] = (
    df_terintegrasi["populasi"] /
    df_terintegrasi["luas_km2"]
)
#Mengelompokkan negara menjadi Pendiri (1967) atau Anggota baru berdasarkan tahun bergabung ASEAN.
df_terintegrasi["kelompok"] = df_terintegrasi[
    "tgl_bergabung_asean"
].apply(
    lambda x: "Pendiri (1967)" if pd.to_datetime(x).year == 1967
    else "Anggota baru"
)

df_terintegrasi.head()


,iso3,tahun,populasi,pdb_per_kapita_usd,internet_persen,nama_negara,ibu_kota,tgl_bergabung_asean,luas_km2,mata_uang,PDB_total_miliar_usd,kepadatan_per_km2,kelompok
0,BRN,2015,422212,30625.13,71.20,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND,12.930297,80.116129,Anggota baru
1,BRN,2016,427564,26663.30,90.00,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND,11.400267,81.131689,Anggota baru
2,BRN,2017,432772,28024.38,94.87,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND,12.128167,82.119924,Anggota baru
3,BRN,2018,437810,30988.12,95.00,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND,13.566909,83.075901,Anggota baru
4,BRN,2019,442680,30426.57,95.00,Brunei Darussalam,Bandar Seri Begawan,1984-01-07,5270,BND,13.469234,84.000000,Anggota baru


## Bagian C — Groupby

 1. Total populasi & PDB Total ASEAN per Tahun

In [ ]:
# Mengelompokkan data berdasarkan 'tahun' dan menghitung total populasi serta total PDB (dalam miliar USD)
ringkasan_tahun = df_terintegrasi.groupby("tahun").agg(
    total_populasi=("populasi", "sum"),
    total_pdb_miliar_usd=("PDB_total_miliar_usd", "sum")
).reset_index()

# Mengambil baris data yang memiliki nilai 'total_pdb_miliar_usd' paling tinggi
tahun_pdb_tertinggi = ringkasan_tahun.loc[
    ringkasan_tahun["total_pdb_miliar_usd"].idxmax()
]

print(ringkasan_tahun)
print(f"Tahun dengan PDB total tertinggi: {int(tahun_pdb_tertinggi['tahun'])}")

   tahun  total_populasi  total_pdb_miliar_usd
0   2015       642385385           2529.936098
1   2016       649533240           2661.790925
2   2017       656442344           2871.905750
3   2018       662968757           3077.009249
4   2019       669237554           3262.263824
5   2020       675054156           3103.353129
6   2021       679873859           3407.610337
7   2022       684885549           3692.280188
8   2023       690245745           3812.283852
9   2024       695353901           4012.918492
Tahun dengan PDB total tertinggi: 2024


2. Analisis Pertumbuhan & Statistik PDB per Kapita (2015 vs 2024)

In [ ]:
# Menyaring data hanya untuk tahun 2015 dan 2024
df_2015_2024 = df_terintegrasi[
    df_terintegrasi["tahun"].isin([2015, 2024])
]

# Mengubah format tabel (pivot) agar PDB per kapita tahun 2015 dan 2024 menjadi kolom berdampingan
pdb_2015_2024 = df_2015_2024.pivot(
    index="nama_negara",
    columns="tahun",
    values="pdb_per_kapita_usd"
).reset_index()

# Mengubah nama kolom angka tahun (2015, 2024) menjadi nama kolom yang lebih informatif
pdb_2015_2024 = pdb_2015_2024.rename(
    columns={
        2015: "pdb_per_kapita_2015",
        2024: "pdb_per_kapita_2024"
    }
)

In [ ]:
# Menghitung nilai minimum dan maksimum PDB per kapita untuk setiap negara selama seluruh rentang tahun
stat_pdb = df_terintegrasi.groupby("nama_negara")[
    "pdb_per_kapita_usd"
].agg(minimum="min", maksimum="max").reset_index()

# Menggabungkan data PDB 2015 & 2024 dengan statistik min/max berdasarkan nama negara
ringkasan_negara = pdb_2015_2024.merge(stat_pdb, on="nama_negara")

In [ ]:
# Menghitung persentase pertumbuhan PDB per kapita dari tahun 2015 ke 2024
ringkasan_negara["pertumbuhan_persen"] = (
    (
        ringkasan_negara["pdb_per_kapita_2024"]
        - ringkasan_negara["pdb_per_kapita_2015"]
    )
    / ringkasan_negara["pdb_per_kapita_2015"]
    * 100
)

In [ ]:
# Mengurutkan hasil dari persentase pertumbuhan tertinggi ke terkecil
ringkasan_negara = ringkasan_negara.sort_values("pertumbuhan_persen",ascending=False)
print(ringkasan_negara)

          nama_negara  pdb_per_kapita_2015  pdb_per_kapita_2024   minimum  \
10            Vietnam              2577.57              4716.66   2577.57   
7           Singapura             55645.61             94896.56  55645.61   
3             Kamboja              1547.32              2627.88   1547.32   
2           Indonesia              3288.22              4925.44   3288.22   
1            Filipina              2909.86              3985.29   2909.86   
8            Thailand              5688.85              7386.64   5688.85   
5            Malaysia              9648.68             11874.43   9476.53   
6             Myanmar              1166.73              1359.05   1158.05   
0   Brunei Darussalam             30625.13             33153.47  26663.30   
9         Timor-Leste              1319.59              1331.97   1219.47   
4                Laos              2121.01              2123.98   2046.40   

    maksimum  pertumbuhan_persen  
10   4716.66           82.988629  
7   9

3. Porsi PDB Kelompok Pendiri vs Anggota Baru

In [ ]:
# Menghitung total PDB per kelompok negara (Pendiri vs Anggota baru) untuk setiap tahun
pdb_kelompok = (
    df_terintegrasi.groupby(["kelompok", "tahun"])
    .agg(pdb_kelompok_miliar_usd=("PDB_total_miliar_usd", "sum"))
    .reset_index()
)

In [ ]:
# Menhitung total PDB keseluruhan negara ASEAN per tahun
total_pdb_tahun = df_terintegrasi.groupby(
    "tahun"
)["PDB_total_miliar_usd"].sum().reset_index()

# Mengubah nama kolom agar jelas sebagai total PDB ASEAN
total_pdb_tahun = total_pdb_tahun.rename(
    columns={
        "PDB_total_miliar_usd":
        "total_pdb_asean_miliar_usd"
    }
)

# Menggabungkan data PDB kelompok dengan total PDB ASEAN berdasarkan tahun
pdb_kelompok = pdb_kelompok.merge(total_pdb_tahun, on="tahun")

In [ ]:
# Menghitung persentase porsi PDB masing-masing kelompok terhadap total PDB ASEAN
pdb_kelompok["porsi_persen"] = (
    pdb_kelompok["pdb_kelompok_miliar_usd"]
    / pdb_kelompok["total_pdb_asean_miliar_usd"]
    * 100
)

In [ ]:
# Menyaring ringkasan porsi PDB hanya untuk tahun 2015 dan 2024
ringkasan_kelompok = pdb_kelompok[pdb_kelompok["tahun"].isin([2015, 2024])]
print(ringkasan_kelompok)

          kelompok  tahun  pdb_kelompok_miliar_usd  \
0     Anggota baru   2015               351.986671   
9     Anggota baru   2024               630.454935   
10  Pendiri (1967)   2015              2177.949427   
19  Pendiri (1967)   2024              3382.463557   

    total_pdb_asean_miliar_usd  porsi_persen  
0                  2529.936098     13.912868  
9                  4012.918492     15.710634  
10                 2529.936098     86.087132  
19                 4012.918492     84.289366  


4. Menyaring Negara dengan Rata-Rata Penetrasi Internet dibawah 50%

In [ ]:
# Menyaring kelompok negara yang memiliki rata-rata penetrasi internet di bawah 50%
ringkasan_internet = df_terintegrasi.groupby(
    "nama_negara"
).filter(
    lambda x: x["internet_persen"].mean() < 50
)

print(ringkasan_internet)

   iso3  tahun   populasi  pdb_per_kapita_usd  internet_persen  nama_negara  \
10  IDN   2015  261799249             3288.22            22.06    Indonesia   
11  IDN   2016  264627418             3521.47            25.45    Indonesia   
12  IDN   2017  267346658             3798.88            32.34    Indonesia   
13  IDN   2018  269951846             3860.95            39.90    Indonesia   
14  IDN   2019  272489381             4106.95            47.69    Indonesia   
15  IDN   2020  274814866             3853.70            53.73    Indonesia   
16  IDN   2021  276758053             4287.17            62.10    Indonesia   
17  IDN   2022  278830529             4730.83            66.48    Indonesia   
18  IDN   2023  281190067             4876.30            69.21    Indonesia   
19  IDN   2024  283487931             4925.44            72.78    Indonesia   
30  LAO   2015    6801645             2121.01            18.20         Laos   
31  LAO   2016    6908802             2303.22       

**Bagian D — Menyimpan Data**

In [ ]:
#Membuat folder output
import os
os.makedirs("output", exist_ok=True)

#Menyimpan data terintegrasi ke CSV
df_terintegrasi.to_csv(
    "output/asean_terintegrasi.csv",
    index=False
)

In [ ]:
#Membuat database SQLite
conn = sqlite3.connect(
    "output/asean_terintegrasi.db"
)

In [ ]:
#Memasukkan DataFrame ke database
df_terintegrasi.to_sql(
    "asean_terintegrasi",
    conn,
    if_exists="replace",
    index=False
)

110

In [ ]:
conn.close() #Menutup koneksi ke database SQLite setelah selesai menyimpan data.

In [ ]:
#Membuat file Excel
with pd.ExcelWriter(
    "output/ringkasan_asean.xlsx"
) as writer:

#Seluruh data hasil integrasi
    df_terintegrasi.to_excel(
        writer,
        sheet_name="Data Terintegrasi",
        index=False
    )

#Hasil Groupby C1
    ringkasan_tahun.to_excel(
        writer,
        sheet_name="C1 PDB Tahunan",
        index=False
    )

#Hasil Groupby C2
    ringkasan_negara.to_excel(
        writer,
        sheet_name="C2 Per Negara",
        index=False
    )

#Hasil Groupby C3
    ringkasan_kelompok.to_excel(
        writer,
        sheet_name="C3 Kelompok",
        index=False
    )

#Hasil Groupby C4
    ringkasan_internet.to_excel(
        writer,
        sheet_name="C4 Internet <50",
        index=False
    )

In [ ]:
import os
os.listdir("output")

['asean_terintegrasi.db', 'asean_terintegrasi.csv', 'ringkasan_asean.xlsx']

Data hasil integrasi telah disimpan dalam format CSV dan database SQLite.
Selain itu, file Excel berisi data terintegrasi dan empat tabel ringkasan dari Bagian C telah berhasil dibuat dalam beberapa sheet.